### Installation

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%capture
# Skip restarting message in Colab
import sys

modules = list(sys.modules.keys())
for x in modules:
    sys.modules.pop(x) if "PIL" in x or "google" in x else None

!pip uninstall -y chromadb langchain langchain-community opentelemetry-sdk
!pip install --no-cache-dir chromadb==0.4.24 langchain-community langchain
!pip install --no-cache-dir llama-cpp-python

In [ ]:
# from langchain.llms import LlamaCpp
from langchain_community.embeddings import HuggingFaceEmbeddings

# import torch
from langchain_community.vectorstores import Chroma
import pandas as pd

# Load Embedding Model
embed_model = HuggingFaceEmbeddings(model_name="Snowflake/snowflake-arctic-embed-l")

### Unsloth

Use `PatchFastRL` before all functions to patch GRPO and other RL algorithms!

### Data Prep
<a name="Data"></a>

Using the yoda stuff


In [ ]:
path = "/content/drive/MyDrive/Misc College/LLMs/data/yoda-corpus.csv"

df = pd.read_csv(path)
df_yoda = df[df["character"] == "YODA"]

df_yoda["text"]

,text
4,"The very Republic is threatened, if involved t..."
6,"Hard to see, the dark side is. Discover who th..."
9,"With this Naboo queen you must stay, Qui-Gon. ..."
11,May the Force be with you.
13,(Cont'd) Master Qui-Gon more to say have you?
...,...
356,Your father he is.
358,"Told you, did he?"
361,"Unexpected this is, and unfortunate..."
367,"Remember, a Jedi's strength flows from the For..."


Tokenizing the yoda dataset


In [ ]:
df_yoda["embedding"] = df_yoda["text"].apply(lambda x: embed_model.embed_query(x))

<ipython-input-20-09ac829433d4>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_yoda["embedding"] = df_yoda["text"].apply(lambda x: embed_model.embed_query(x))


Embeddings

In [ ]:
# Initialize ChromaDB with Hugging Face embeddings
vector_store = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)

# Store Embeddings
for i, row in df_yoda.iterrows():
    vector_store.add_texts(
        texts=[row["text"]],
        metadatas=[{"character": "YODA"}],
        ids=[str(i)],
        embeddings=[row["embedding"]],
    )

vector_store.persist()

In [ ]:
def query_vector_db(query, k=3):
    query_embedding = embed_model.embed_query(query)
    results = vector_store.similarity_search_by_vector(query_embedding, k)
    return [res.page_content for res in results]


# Example Query
query_result = query_vector_db("Tell me about the dangerous Sith")
print(query_result)

[' Destroy the Sith, we must.', 'The very Republic is threatened, if involved the Sith are.', ' To fight this Lord Sidious, strong enough, you are not.']
